# pyLSTemp - Exemplo de `water_vapor(...)`

Este notebook demonstra como estimar vapor d'agua precipitavel com `water_vapor(...)`.

O metodo atual (`wang-2015`) usa brightness das bandas termais 10 e 11 e uma imagem NDVI.

## 1. Instalacao

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importacoes

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import brightness, spectral_index, water_vapor, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir metodos de vapor d'agua

In [ ]:
catalog = list_algorithms()
print(list(catalog["water_vapor"].keys()))

## 4. Ler bandas necessarias

O metodo precisa de:

- Brightness Band 10.
- Brightness Band 11.
- NDVI.

In [ ]:
data_dir = Path("data")

band_10_path = data_dir / "LC82210712016107LGN01_B10.tif"
band_11_path = data_dir / "LC82210712016107LGN01_B11.tif"
red_path = data_dir / "LC82210712016107LGN01_B4.tif"
nir_path = data_dir / "LC82210712016107LGN01_B5.tif"

with rasterio.open(band_10_path) as src:
    band_10 = src.read(1).astype(float)
    profile = src.profile.copy()

with rasterio.open(band_11_path) as src:
    band_11 = src.read(1).astype(float)

with rasterio.open(red_path) as src:
    red = src.read(1).astype(float)

with rasterio.open(nir_path) as src:
    nir = src.read(1).astype(float)

print("Shape:", band_10.shape)

## 5. Calcular brightness e NDVI

In [ ]:
mask_10 = band_10 == 0
mask_11 = band_11 == 0
mask_optical = (red == 0) | (nir == 0)

brightness_10 = brightness(band_10, band="band_10", sensor="landsat_8", mask=mask_10)
brightness_11 = brightness(band_11, band="band_11", sensor="landsat_8", mask=mask_11)
ndvi_image = spectral_index(index="ndvi", nir=nir, red=red, mask=mask_optical)

print("Brightness 10 mean:", np.nanmean(brightness_10))
print("Brightness 11 mean:", np.nanmean(brightness_11))
print("NDVI mean:", np.nanmean(ndvi_image))

## 6. Calcular vapor d'agua

`window_size=5` significa uma janela local 5 x 5 ao redor de cada pixel. O valor deve ser impar e maior ou igual a 3.

`group_count=5` divide os pixels validos da janela em 5 grupos de NDVI antes de estimar o vapor d'agua.

In [ ]:
water_vapor_image = water_vapor(
    brightness_band_10=brightness_10,
    brightness_band_11=brightness_11,
    ndvi_image=ndvi_image,
    method="wang-2015",
    window_size=5,
    group_count=5,
)

print("Water vapor min:", np.nanmin(water_vapor_image))
print("Water vapor max:", np.nanmax(water_vapor_image))
print("Water vapor mean:", np.nanmean(water_vapor_image))

## 7. Visualizar resultado

In [ ]:
valid = water_vapor_image[np.isfinite(water_vapor_image)]
vmin, vmax = np.nanpercentile(valid, [2, 98])

plt.figure(figsize=(8, 6))
plt.imshow(water_vapor_image, cmap="Blues", vmin=vmin, vmax=vmax)
plt.colorbar(label="Water vapor (g/cm2)")
plt.title("Water vapor - Wang 2015")
plt.axis("off")
plt.show()

## 8. Usar em split-window

O resultado pode ser passado para metodos que exigem vapor d'agua, por exemplo:

`split_window(..., lst_method="jimenez-munoz-2014", water_vapor=water_vapor_image)`